# 01  Analysis of club member data

1)  Imputation of missing data

In [47]:
import numpy as np

In [48]:
gym_data = np.array([
    [28, 75, 175, 4],
    [34, 68, 168, 3],
    [45, 82, 180, 2],
    [22, 58, 162, 5],
    [38, 90, 0, 1],
    [29, 65, 170, 0]
], dtype=float)

In [49]:
member_names = np.array([
    "Ali", "Sara", "Reza", "Neda", "Hassan", "Maryam"
])


In [50]:
weight_mask = gym_data[:, 1] == 0
gym_data[weight_mask, 1] = np.mean(gym_data[~weight_mask, 1])

height_mask = gym_data[:, 2] == 0
gym_data[height_mask, 2] = np.mean(gym_data[~height_mask, 2])

sessions_mask = gym_data[:, 3] == 0
gym_data[sessions_mask, 3] = np.mean(gym_data[~sessions_mask, 3])

In [107]:
print(gym_data)

[[ 28.          75.         175.           4.          14.24489796]
 [ 34.          68.         168.           3.          13.54648526]
 [ 45.          82.         180.           2.          13.65432099]
 [ 22.          58.         162.           5.          13.5501448 ]
 [ 38.          90.         171.           1.          15.88935057]
 [ 29.          65.         170.           3.          12.74567474]]


2) Creating the weight, height, and sessions array

In [52]:
gym_features = gym_data[:, 1:4]

print(gym_features)
print(gym_features.shape)

[[ 75. 175.   4.]
 [ 68. 168.   3.]
 [ 82. 180.   2.]
 [ 58. 162.   5.]
 [ 90. 171.   1.]
 [ 65. 170.   3.]]
(6, 3)


3) Calculation of physical fitness score

In [53]:
weight = gym_data[:, 1]
height_m = gym_data[:, 2] / 100
sessions = gym_data[:, 3]

bmi = weight / (height_m ** 2)

beta1 = 0.5
beta2 = 0.5
fitness_score = beta1 * bmi + beta2 * sessions
gym_data = np.column_stack((gym_data, fitness_score))
print(np.round(gym_data, 2))

[[ 28.    75.   175.     4.    14.24]
 [ 34.    68.   168.     3.    13.55]
 [ 45.    82.   180.     2.    13.65]
 [ 22.    58.   162.     5.    13.55]
 [ 38.    90.   171.     1.    15.89]
 [ 29.    65.   170.     3.    12.75]]


4) Finding the member with the highest score

In [54]:
best_idx = np.argmax(fitness_score)

print("Best member:", member_names[best_idx])
print("Fitness score:", round(fitness_score[best_idx], 2))

Best member: Hassan
Fitness score: 15.89


5) Finding the difference in the highest number of sessions

In [55]:
session_std = np.std(sessions)

print("Session standard deviation:", round(session_std, 2))

Session standard deviation: 1.29


# 02 Food recommendation system using Euclidean distance

1) Normalizing Food Features Using Min-Max Scaling

In [56]:
recipes = np.array([
    [15, 350, 2, 5],
    [45, 600, 7, 10],
    [10, 200, 0, 3],
    [30, 450, 5, 7],
    [60, 800, 8, 12]
], dtype=float)


In [57]:
recipe_names = np.array([
    "Salad", "Curry", "Toast", "Pasta", "Stew"
])

In [58]:
mins = recipes.min(axis=0)
maxs = recipes.max(axis=0)

recipes_scaled = (recipes - mins) / (maxs - mins)

In [59]:
print("Minimum:", mins)
print("Maximum:", maxs)
print(np.round(recipes_scaled, 3))

Minimum: [ 10. 200.   0.   3.]
Maximum: [ 60. 800.   8.  12.]
[[0.1   0.25  0.25  0.222]
 [0.7   0.667 0.875 0.778]
 [0.    0.    0.    0.   ]
 [0.4   0.417 0.625 0.444]
 [1.    1.    1.    1.   ]]


2) Normalizing user preferences using the same previous values.

In [60]:
users = np.array([
    [10, 250, 1, 4],
    [50, 700, 8, 11],
    [25, 400, 4, 6]
], dtype=float)


In [61]:
users_scaled = (users - mins) / (maxs - mins)

In [62]:
print(np.round(users_scaled, 3))

[[0.    0.083 0.125 0.111]
 [0.8   0.833 1.    0.889]
 [0.3   0.333 0.5   0.333]]


3) Calculating Euclidean distance without a for-loop

In [63]:
diff = users_scaled[:, np.newaxis, :] - recipes_scaled[np.newaxis, :, :]
distances = np.linalg.norm(diff, axis=2)

print("Shape:", distances.shape)
print(np.round(distances, 3))

Shape: (3, 5)
[[0.256 1.355 0.187 0.795 1.843]
 [1.355 0.256 1.768 0.82  0.283]
 [0.349 0.781 0.75  0.212 1.276]]


4) The nearest food option for each user

In [64]:
nearest_idx = np.argmin(distances, axis=1)
for_user = recipe_names[nearest_idx]

for i, name in enumerate(for_user):
    print(f"User {i + 1}: {name}")

User 1: Toast
User 2: Curry
User 3: Pasta


5) Sorting all foods from nearest to farthest

In [65]:
rankings = np.argsort(distances, axis=1)

for i, row in enumerate(rankings):
    print(f"User {i + 1}:")
    print(recipe_names[row])

User 1:
['Toast' 'Salad' 'Pasta' 'Curry' 'Stew']
User 2:
['Curry' 'Stew' 'Pasta' 'Salad' 'Toast']
User 3:
['Pasta' 'Salad' 'Toast' 'Curry' 'Stew']


# 03 Various scoring methods

1) Constructing the weight matrix

In [66]:
scores = np.array([
    [18, 15, 20],
    [12, 14, 16],
    [20, 19, 18],
    [10, 8, 15]
])

In [67]:
scheme_A = np.array([0.5, 0.3, 0.2])
scheme_B = np.array([0.2, 0.3, 0.5])
scheme_C = np.array([0.1, 0.2, 0.7])


In [68]:
weight_matrix = np.vstack([
    scheme_A,
    scheme_B,
    scheme_C
])

In [69]:
print(weight_matrix)
print(weight_matrix.shape)

[[0.5 0.3 0.2]
 [0.2 0.3 0.5]
 [0.1 0.2 0.7]]
(3, 3)


2) Calculating the final score using matrix multiplication

In [70]:
final_scores = scores @ weight_matrix.T

print(final_scores)
print(final_scores.shape)

[[17.5 18.1 18.8]
 [13.4 14.6 15.2]
 [19.3 18.7 18.4]
 [10.4 11.9 13.1]]
(4, 3)


3) The best method for every student

In [71]:
best_scheme = np.argmax(final_scores, axis=1)

print(best_scheme)

[2 2 0 2]


4) Selecting the appropriate method for the entire class

In [72]:
class_means = final_scores.mean(axis=0)
print("Class means:", class_means)

best_class_scheme = np.argmax(class_means)
print("Best class scheme:", best_class_scheme)

Class means: [15.15  15.825 16.375]
Best class scheme: 2


5) Adding Method D, with a final exam weight double that of the quiz.

In [73]:
scheme_D = np.array([0.25, 0.25, 0.50])

weight_matrix = np.vstack([
    scheme_A,
    scheme_B,
    scheme_C,
    scheme_D
])



In [74]:
print(weight_matrix)
print(weight_matrix.shape)

[[0.5  0.3  0.2 ]
 [0.2  0.3  0.5 ]
 [0.1  0.2  0.7 ]
 [0.25 0.25 0.5 ]]
(4, 3)


In [75]:
final_scores = scores @ weight_matrix.T

print(final_scores)
print(final_scores.shape)

[[17.5  18.1  18.8  18.25]
 [13.4  14.6  15.2  14.5 ]
 [19.3  18.7  18.4  18.75]
 [10.4  11.9  13.1  12.  ]]
(4, 4)


# 04 Analysis of sensor data and transposition error

1) Day 2 extraction and array shape transformation

In [76]:

weather_data = np.random.randn(2, 8, 4) * 5 + 20

def analyze_day(day):
    print("Input shape:", day.shape)

day2 = weather_data[1]
day2 = day2.T
analyze_day(day2)

Input shape: (4, 8)


2) Converting all data into a one-dimensional array

In [77]:
weather_flat = weather_data.reshape(-1)

print(weather_flat.shape)

(64,)


# or

In [78]:
weather_flat = weather_data.flatten()

print(weather_flat.shape)

(64,)


3) Why are "transpose" and "flatten" not equivalent?

In [79]:
A = np.array([
    [1, 2],
    [3, 4]
])


In [80]:

print("Original:")
print(A)

print("Transpose:")
print(A.T)

print("Original flattened:")
print(A.flatten())

print("Transpose flattened:")
print(A.T.flatten())

Original:
[[1 2]
 [3 4]]
Transpose:
[[1 3]
 [2 4]]
Original flattened:
[1 2 3 4]
Transpose flattened:
[1 3 2 4]


4) Adding the third day without manually modifying the initial array.

In [81]:
day3 = np.random.randn(8, 4) * 5 + 20
weather_data = np.concatenate(
    (weather_data, day3[np.newaxis, :, :]),
    axis=0
)

print(weather_data.shape)

(3, 8, 4)


# 05 Smart Note-Taking and View/Copy Behavior

1) Converting an array to a column format using two methods

In [82]:
importance = np.array([3, 8, 1, 9, 4, 7])
#1
column1 = importance.reshape(6, 1)
#or
#2
column2 = importance[:, np.newaxis]

print(column1.shape)
print(column2.shape)

(6, 1)
(6, 1)


2)  Extracting the value 0.9 from the 3D array.

In [83]:
model_out = np.array([[[0.9]]])
#1
value1 = model_out.item()
#or
#2
value2 = model_out[0, 0, 0]

print(value1)
print(value2)

0.9
0.9


3) Predicting the output of the View code

In [84]:
notes = np.array([5, 10, 15, 20, 25])
pinned = notes[1:3]
pinned[0] = 999

print(notes)

[  5 999  15  20  25]


In [85]:
pinned = notes[1:3].copy()
pinned[0] = 999

print("notes:", notes)
print("pinned:", pinned)

notes: [  5 999  15  20  25]
pinned: [999  15]


4) Deliberately creating a view to modify the original array

In [86]:
notes = np.array([5, 10, 15, 20, 25])
pinned = notes[1:3]
pinned[1] = 777

print("notes:", notes)
print("pinned:", pinned)

notes: [  5  10 777  20  25]
pinned: [ 10 777]


# 06 5km running race

1) Adding new runners

In [87]:
bib_numbers = np.array([101, 102, 103, 104, 105, 106])
times_5k = np.array([22.3, 25.1, 21.8, 26.4, 23.0, 24.7])


In [92]:
bib_numbers_2 = np.array([107, 108])
times_5k_2 = np.array([20.5, 27.9])


In [94]:
bib_numbers = np.concatenate((bib_numbers, bib_numbers_2))
times_5k = np.concatenate((times_5k, times_5k_2))

In [95]:
print(bib_numbers)
print(times_5k)

[101 102 103 104 105 106 107 108 107 108]
[22.3 25.1 21.8 26.4 23.  24.7 20.5 27.9 20.5 27.9]


2) Sorting runners by time

In [ ]:
order = np.argsort(times_5k)
sorted_bibs = bib_numbers[order]
sorted_times = times_5k[order]

print(sorted_bibs)
print(sorted_times)

[107 107 103 101 105 106 102 104 108 108]
[20.5 20.5 21.8 22.3 23.  24.7 25.1 26.4 27.9 27.9]


3) Print ranking table

In [97]:
print("Rank | Bib | Time (min)")

for rank, idx in enumerate(order, start=1):
    print(f"{rank:4} | {bib_numbers[idx]:3} | {times_5k[idx]:.1f}")

Rank | Bib | Time (min)
   1 | 107 | 20.5
   2 | 107 | 20.5
   3 | 103 | 21.8
   4 | 101 | 22.3
   5 | 105 | 23.0
   6 | 106 | 24.7
   7 | 102 | 25.1
   8 | 104 | 26.4
   9 | 108 | 27.9
  10 | 108 | 27.9


4) Finding the exact rank of runner number 104

In [98]:
idx_104 = np.where(sorted_bibs == 104)[0][0]
rank_104 = idx_104 + 1

print("Rank of bib 104:", rank_104)

Rank of bib 104: 8


# 07  uilding a simple neuron to detect a signal

1) Calculation of the neuron's raw output

In [99]:
X_messages = np.array([
    [12, 0, 1],
    [45, 5, 8],
    [8, 0, 0],
    [30, 3, 4]
])

In [100]:
w = np.array([0.1, 0.8, 0.5])
b = -2.0
z = X_messages @ w + b

print(z)

[-0.3 10.5 -1.2  5.4]


2) Preventing negative output using the ReLU function

In [101]:
final_output = np.maximum(0, z)

print(final_output)

[ 0.  10.5  0.   5.4]


3) Categorizing messages using a threshold

In [102]:
threshold = 5
is_energetic = final_output >= threshold

for i, result in enumerate(is_energetic, start=1):
    label = "Energetic" if result else "Calm"
    print(f"Message {i}: {label}")

Message 1: Calm
Message 2: Energetic
Message 3: Calm
Message 4: Energetic
